# Example: Double Pendulum

This notebook is the supported interactive entrypoint for `examples/double_pendulum`.
It uses the current `phases` config shape and applies a small `config_mod` overlay so the run stays notebook-friendly.

The `deprecated/` folder is kept only for historical comparison material.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

from dymad.io import load_model
from dymad.models import KBF
from dymad.training import WeakFormTrainer
from dymad.utils import TrajectorySampler, plot_summary

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)

cwd = Path.cwd().resolve()
if (cwd / "examples" / "double_pendulum").exists():
    EXAMPLE_DIR = cwd / "examples" / "double_pendulum"
elif cwd.name == "double_pendulum":
    EXAMPLE_DIR = cwd
else:
    raise RuntimeError("Run this notebook from the repo root or from examples/double_pendulum.")

os.chdir(EXAMPLE_DIR)
DATA_DIR = EXAMPLE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

print(f"Working directory: {EXAMPLE_DIR}")


## Data Generation

We keep the full double-pendulum dynamics, but use a smaller dataset than the tracked YAML defaults so the notebook can be rerun quickly.


In [ ]:
BATCH = 16
N_STEPS = 151
t_grid = np.linspace(0.0, 5.0, N_STEPS)
DATA_PATH = DATA_DIR / "dp_demo.npz"


def f(t, x, u):
    dx = np.zeros_like(x)
    m1, m2 = 1.0, 1.0
    l1, l2 = 1.0, 1.0
    g = 9.81
    th1, th2, w1, w2 = x[0], x[1], x[2], x[3]

    dx[0] = w1
    dx[1] = w2
    dx[2] = (
        -g * (2 * m1 + m2) * np.sin(th1)
        - m2 * g * np.sin(th1 - 2 * th2)
        - 2 * np.sin(th1 - th2) * m2 * (l2 * w2**2 + l1 * w1**2 * np.cos(th1 - th2))
    ) / (l1 * (2 * m1 + m2 - m2 * np.cos(2 * th1 - 2 * th2)))
    dx[3] = (
        2
        * np.sin(th1 - th2)
        * (
            l1 * w1**2 * (m1 + m2)
            + g * (m1 + m2) * np.cos(th1)
            + l2 * w2**2 * m2 * np.cos(th1 - th2)
        )
    ) / (l2 * (2 * m1 + m2 - m2 * np.cos(2 * th1 - 2 * th2)))
    return dx


def g(t, x, u):
    return x


sampler = TrajectorySampler(f, g, config="double_pendulum_data.yaml", rng=SEED)
ts, xs, us, ys = sampler.sample(t_grid, batch=BATCH, save=str(DATA_PATH))

print(f"Saved dataset to {DATA_PATH}")
print(f"t shape: {ts.shape}, x shape: {xs.shape}, u shape: {us.shape}, y shape: {ys.shape}")


In [ ]:
labels = ["theta_1", "theta_2", "omega_1", "omega_2"]
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for i, ax in enumerate(axes.flat):
    ax.plot(ts[0], ys[0, :, i], color="black", linewidth=1.5)
    ax.set_title(labels[i])
    ax.set_xlabel("time")
    ax.grid(alpha=0.2)
plt.tight_layout()


## Current Config Shape

The tracked example configs use explicit `phases` entries. For the notebook run, we keep the same surface and only override a few values through `config_mod`.


In [ ]:
with open("dp_kbf_wf.yaml", encoding="utf-8") as handle:
    base_config = yaml.safe_load(handle)

base_config["phases"]


In [ ]:
config_mod = {
    "data": {
        "path": str(DATA_PATH),
        "n_samples": BATCH,
        "n_steps": N_STEPS,
        "split_seed": SEED,
    },
    "model": {
        "name": "dp_kbf_wf_demo",
        "hidden_dimension": 16,
        "koopman_dimension": 8,
    },
    "dataloader": {"batch_size": 16},
    "phases": [
        {
            "type": "optimizer",
            "name": "weak_form",
            "trainer": "Weak",
            "n_epochs": 5,
            "save_interval": 1,
            "load_checkpoint": False,
            "learning_rate": 1e-2,
            "decay_rate": 0.999,
            "reconstruction_weight": 1.0,
            "dynamics_weight": 1.0,
            "weak_form_params": {
                "N": 13,
                "dN": 2,
                "ordpol": 2,
                "ordint": 2,
            },
        }
    ],
}

torch.manual_seed(SEED)
trainer = WeakFormTrainer("dp_kbf_wf.yaml", KBF, config_mod=config_mod)
_ = trainer.train()


In [ ]:
_ = plot_summary(["dp_kbf_wf_demo"], labels=["KBF / weak form"], ifclose=False)


## Prediction Check

Load the trained checkpoint and compare it against one fresh trajectory sampled with a different RNG seed.


In [ ]:
eval_sampler = TrajectorySampler(f, g, config="double_pendulum_data.yaml", rng=SEED + 1)
t_eval, x_eval, u_eval, y_eval = eval_sampler.sample(t_grid, batch=1)

_, predict_fn = load_model(KBF, "dp_kbf_wf_demo/dp_kbf_wf_demo.pt")
with torch.no_grad():
    pred = predict_fn(x_eval[0], t_eval[0], u=u_eval[0])

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for i, ax in enumerate(axes.flat):
    ax.plot(t_eval[0], x_eval[0, :, i], label="truth", color="black", linewidth=1.5)
    ax.plot(t_eval[0], pred[:, i], "--", label="prediction", color="#c0392b", linewidth=1.5)
    ax.set_title(labels[i])
    ax.set_xlabel("time")
    ax.grid(alpha=0.2)
axes[0, 0].legend(loc="best")
plt.tight_layout()
